# Trelis Whisper Fine-tuning (v2.0.0 - unsloth)

Purchase a copy of this notebook at [Trelis.com](https://trelis.com/ADVANCED-transcription).

Notebook built by Trelis Research. Find us at [Trelis.com](https://trelis.com/About), [YouTube](https://YouTube.com/@TrelisResearch) and on [HuggingFace](https://huggingface.co/Trelis).

*Subscribe to Trelis Research emails [here](https://trelis.substack.com) and get notified each time a new video tutorial is published.*

Built upon an [original transformers notebook](https://colab.research.google.com/drive/1DOkD_5OUjFa0r5Ik3SgywJLJtEo2qLxO?usp=sharing#scrollTo=8d230e6d-624c-400a-bbf5-fa660881df25) by HuggingFace and an [original unsloth notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Whisper.ipynb).

## Installation

In [1]:
import subprocess

# Run the command and capture the output
add_repo_output = subprocess.run(['add-apt-repository', '-y', 'ppa:jonathonf/ffmpeg-4'], capture_output=True, text=True)
update_output = subprocess.run(['apt', 'update', '-q'], capture_output=True, text=True)
install_ffmpeg_output = subprocess.run(['apt', 'install', '-y', 'ffmpeg', '-q'], capture_output=True, text=True)

# # Print the output from each command
# print("Add repository output:")
# print(add_repo_output.stdout)

# print("Update output:")
# print(update_output.stdout)

# print("Install FFmpeg output:")
# print(install_ffmpeg_output.stdout)

In [2]:
# Install dependencies with compatible versions
%cd /workspace/ADVANCED-transcription
# !uv pip uninstall hf_transfer datasets transformers unsloth
!uv pip install librosa evaluate jiwer ipywidgets webvtt-py setuptools whisper-timestamped -qU

/workspace/ADVANCED-transcription


In [3]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !uv pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !uv pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !uv pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !uv pip install --no-deps unsloth
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2
!uv pip install torchcodec

In [4]:
# allow for fast weight uploads and downloads
%cd /workspace/ADVANCED-transcription/speech-to-text
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

/workspace/ADVANCED-transcription/speech-to-text


In [9]:
# Run if not using the Trelis Runpod template that will already have injected your hf_token
# from huggingface_hub import notebook_login

# notebook_login()

----
Video Layout:

Goal: Fine-tune on a custom dataset, namely AI terms.

1. Record some audio 
2. Transcribe the audio
3. Clean the transcription (LLM or human)
4. Fine-tune on that data
5. Make the model ready for inference, including via a continuous batching server

In [11]:
import os

# Select CUDA device index - this just let's the script know we want to use one gpu (the 0th one).
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Input model
transcription_model_name_or_path = "unsloth/whisper-large-v3"
model_name_or_path = "unsloth/whisper-large-v3-turbo" # this will use more vram than whisper-small but is higher quality (and 2x faster at inference)
language = "English"
language_abbr = "en"
task = "transcribe"

# Set this to pre-load a dataset OR as the repo to push a generated dataset to
dataset_name = "Trelis/llm-lingo"

# Output model path (for pushing to HuggingFace)
org = "Trelis"
trained_model_name = "llm-lingo"
trained_adapter_name = f"{trained_model_name}-adapters"

trained_adapter_repo = org + "/" + trained_adapter_name
trained_model_repo = org + '/' + trained_model_name

## Dataset Generation (Skip if you already have a dataset)

You'll now upload train.mp3 and validation.mp3 files (or train-xyz.mp3 and validation-xyz.mp3 files). At least 30 seconds of audio is recommended for each.

You'll then run the cells below to generate train.vtt and validation.vtt transcripts. Those transcripts will need to be manually corrected (optionally, with the help of a GPT).

Note that we're not using unsloth here, we're inferencing whisper using transformers, to transcribe the audio.

You could try to manually transcribe, however, this can bring you out of distribution on a pre-trained model. So it's better to transcribe and correct.

In [12]:
train_audio_filename ="train"
validation_audio_filename = "validation"

Transcribing and assembling chunks

1. Start with transcribing word chunks
2. Add word chunks to form segments until I hit at least 20 seconds
3. Close the segment if I reach a full stop (;:!?) OR a long pause
4. If I hit 30 seconds without a pause or full stop, then force the end.

In [7]:
# These imports are for the training section later
# from transformers import pipeline
from transformers import (
    WhisperForConditionalGeneration,
    WhisperTokenizer,
    WhisperProcessor,
    pipeline
)
import torch

In [13]:
# FOR DEMONSTRATION PURPOSES ONLY - WHISPER PIPELINE
if True:
    # # These two lines allow the first tokens of the output to be forced as the language and task - which accelerates transcription.
    # # To force the ids, generate_kwargs needs to be passed to the pipeline
    # # This is unnecessary if the model is capable for determining the language and task.
    # # Fine-tuning can adversely affect a model's ability to determine the language.
    # processor = WhisperProcessor.from_pretrained(model_name_or_path, language=language, task=task)
    # forced_decoder_ids = processor.get_decoder_prompt_ids(language=language, task=task)

    whisper_asr = pipeline(
        "automatic-speech-recognition",
        model=transcription_model_name_or_path,
        chunk_length_s=30,
        device="cuda" if torch.cuda.is_available() else "mps", # for mac
        # device="cuda" if torch.cuda.is_available() else "cpu",
        return_timestamps="word",
    )

    # Test out transcription on the base model
    prediction = whisper_asr(
        "data/train.mp3",
        return_timestamps=True, # True means segment timestamps; "word" means word timestamps.
        chunk_length_s=30,
    )

    def print_chunks(prediction):
        print("\nFULL TEXT:\n")
        print(prediction["text"])
        print("\n\nCHUNKS:\n")

        for i, ch in enumerate(prediction.get("chunks", []), start=1):
            start, end = ch["timestamp"]
            text = ch["text"].strip()

            print(f"--- Chunk {i} ---")
            print(f"Start: {start:8.2f}s   End: {end:8.2f}s   (len={end-start:.2f}s)")
            print(text)
            print()

    print_chunks(prediction)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

Device set to use cuda
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription


FULL TEXT:

 Spin. Self-play fine-tuning that improves LLMs. Trixie. It's a form of fast inference involving sparsity. PHY2 is a model from Microsoft. Lightning Attention 2 is an alternative to Flash Attention. Mixtral 8x7B. This is a mixture of Experts model. Solar 10.7Bb is a mistral model with some extra layers added in open chat is a fine tune of the mistral model notux 8x7b v1 this is a version of the mistral model fine-tuned gemini pro is google's best model or perhaps not as good as gemini I'll see you next time. a version of the mixtral model fine-tuned gemini pro is google's best model or perhaps not as good as gemini ultra microsoft fi 2 i've already mentioned desi lm is a high speed 7b model that's desi lm 7b arena elo is a means of comparing llm's empty bench is another metric and mmlu is also gpt4 turbo is a fast gpt4 model by opening eye mistral medium is a mixture of experts but with larger experts then mixed tral 8x7b7B, Cloud 1, Cloud 2, or Cloud 2.0 are the latest Cl

In [14]:
import whisper_timestamped as whisper

# Load the Whisper model for transcription
whisper_model = whisper.load_model(transcription_model_name_or_path, device="cuda" if torch.cuda.is_available() else "cpu")

Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



In [19]:
if True:
    # Test out transcription with whisper-timestamped
    audio = whisper.load_audio("data/train.mp3")
    result = whisper.transcribe(whisper_model, audio, language="en")
    
    print("\nFULL TEXT:\n")
    print(result["text"])
    print("\n\nWORD TIMESTAMPS (first 10):\n")
    
    word_count = 0
    for segment in result.get("segments", []):
        for word_info in segment.get("words", []):
            start = word_info["start"]
            end = word_info["end"]
            text = word_info["text"]
            print(f"{text:20s} {start:8.2f}s -> {end:8.2f}s")
            word_count += 1
            if word_count >= 10:
                break
        if word_count >= 10:
            break
    
    print(f"\n... (showing first 10 of {sum(len(seg.get('words', [])) for seg in result.get('segments', []))} total words)")

100%|██████████| 13623/13623 [00:18<00:00, 740.81frames/s]


FULL TEXT:

 Spin. Self play, fine tuning that improves LLMs. Trixie. It's a form of fast inference involving sparsity. PHY2 is a model from Microsoft. Lightning Attention 2 is an alternative to Flash Attention. Mixtral 8x7B. This is a mixture of Expert's model. Solar 10.7B is a Mixtral model with some extra layers added in. OpenChat is a fine tune of the Mixtral model. Nultux 8x7B V1. This is a version of the Mixtral model, fine tuned. Gemini Pro is Google's best model, or perhaps not as good as Gemini Ultra. Microsoft PHY2, I've already mentioned. Desi LM is a high speed 7B model. That's Desi LM 7B. Arena Elo is a means of comparing LLMs. MT Bench is another metric and MM LU is also. GPT-4 Turbo is a fast GPT-4 model by OpenAI. Mixtral Medium is a mixture of Expert's but with larger Expert's than Mixtral 8x7B. Clod 1. Clod 2. Clod 2.0 are the latest Clod models. Mixtral 8x7B Instruct V1, or rather V01. That's Mixtral 8x7B Instruct V0.1. That's the latest mixture of Expert's. Yi34B C

In [16]:
import re
import whisper_timestamped as whisper

def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = seconds % 60

    # Clamp edge case like 59.9995 rounding to 60.000
    if secs >= 60:
        secs -= 60
        minutes += 1
        if minutes >= 60:
            minutes -= 60
            hours += 1

    return f"{hours:02}:{minutes:02}:{secs:06.3f}"


def write_vtt_cue(
    vtt_file, seg_start, seg_end, words, cue_index=None,
    max_chunk_s=30.0,
    pad_before=0.2,
    pad_after=0.25,
):
    """Helper to write a single VTT cue with padding, bounded safely."""
    
    if not words:
        return

    # Apply padding
    padded_start = max(seg_start - pad_before, 0)
    padded_end = seg_end + pad_after

    duration = padded_end - padded_start
    if duration <= 0:
        print(f"Skipping segment with non-positive duration: {seg_start}–{seg_end}")
        return

    # This should never happen with proper word-level splitting
    if duration > max_chunk_s:
        print(f"[ERROR] Segment duration {duration:.3f}s exceeds max {max_chunk_s}s after padding!")
        print(f"  Segment: {seg_start:.2f}s -> {seg_end:.2f}s, padded: {padded_start:.2f}s -> {padded_end:.2f}s")
        print(f"  This is a bug - segments should be split before exceeding the limit")
        # Emergency: shrink padding to fit, but this shouldn't happen
        excess = duration - max_chunk_s
        padded_end -= excess
        if padded_end <= padded_start:
            padded_end = padded_start + 0.001

    start_time = format_time(padded_start)
    end_time = format_time(padded_end)

    text = " ".join(w.strip() for w in words).strip()
    if not text:
        return

    vtt_file.write(f"{start_time} --> {end_time}\n{text}\n\n")


def process_audio_and_create_vtt(
    audio_filename,
    audio_type,
    whisper_model,
    output_filename=None,
    target_chunk_s=20.0,   # aim for ~20s-30s segments
    max_chunk_s=30.0,      # hard cap at 30s
    pause_boundary_s=3.0,  # optional: treat long pauses as phrase breaks
    pad_before=0,        # padding to add before segment
    pad_after=0,        # padding to add after segment
):
    # Load audio using whisper's built-in loader
    audio = whisper.load_audio(f"{audio_filename}.{audio_type}")
    
    # Transcribe with whisper-timestamped
    result = whisper.transcribe(whisper_model, audio, language="en")
    
    # Convert whisper-timestamped format to the format expected by VTT code
    # whisper-timestamped returns segments with words, we need to flatten to word list
    chunks = []
    for segment in result.get("segments", []):
        for word_info in segment.get("words", []):
            chunks.append({
                "text": word_info["text"],
                "timestamp": (word_info["start"], word_info["end"])
            })
    
    print(f"DEBUG: Total words transcribed: {len(chunks)}")

    vtt_file_name = output_filename if output_filename else f"data/raw/{audio_filename.split('/')[-1]}-raw.vtt"

    # Calculate the maximum raw duration (before padding) to ensure we never exceed max_chunk_s after padding
    max_raw_duration = max_chunk_s - pad_before - pad_after

    def is_phrase_terminal(word_text: str) -> bool:
        # Heuristic: treat end punctuation as phrase boundaries
        return word_text.endswith((".", "?", "!", ";", ":"))

    with open(vtt_file_name, "w", encoding="utf-8") as vtt_file:
        vtt_file.write("WEBVTT\n\n")

        cue_index = 1

        # Current segment (what will become one VTT cue)
        seg_start = None
        seg_end = None
        seg_words = []

        prev_end = None

        for i, chunk in enumerate(chunks):
            start, end = chunk.get("timestamp", (None, None))
            word = chunk.get("text", "").strip()

            # Basic validation
            if start is None or end is None or not word:
                print(f"Skipping invalid word-chunk at {i}: start={start}, end={end}, text='{word}'")
                continue
            if end <= start:
                print(f"Skipping non-positive duration word at {i}: start={start}, end={end}")
                continue

            word_duration = end - start
            if word_duration > max_chunk_s:
                # Pathological case: a single word spans >max_chunk_s in timestamps.
                print(f"Skipping single word > {max_chunk_s}s at {i}: duration={word_duration:.3f}s")
                continue

            # Check if we should start a new segment
            should_start_new = False
            
            # Reason 1: No current segment
            if seg_start is None:
                seg_start = start
                seg_end = end
                seg_words = [word]
                prev_end = end
                continue
            
            approaching_target = (end - seg_start) > target_chunk_s
            if approaching_target:
                # Reason 2: Prefer breaking at phrase boundaries when approaching target length
                is_phrase_end = is_phrase_terminal(seg_words[-1]) if seg_words else False
                if is_phrase_end:
                    should_start_new = True
                # Reason 3: Long pause detected
                elif prev_end is not None and start - prev_end >= pause_boundary_s:
                    should_start_new = True
                # Reason 4: Adding this word would exceed max duration
                elif end - seg_start > max_raw_duration:
                    should_start_new = True
            
            if should_start_new:
                # Flush current segment
                write_vtt_cue(
                    vtt_file,
                    seg_start,
                    seg_end,
                    seg_words,
                    cue_index=cue_index,
                    max_chunk_s=max_chunk_s,
                    pad_before=pad_before,
                    pad_after=pad_after,
                )
                cue_index += 1
                
                # Start new segment with current word
                seg_start = start
                seg_end = end
                seg_words = [word]
            else:
                # Add word to current segment
                seg_words.append(word)
                seg_end = end
            
            prev_end = end

        # Flush final segment
        if seg_words:
            write_vtt_cue(
                vtt_file,
                seg_start,
                seg_end,
                seg_words,
                cue_index=cue_index,
                max_chunk_s=max_chunk_s,
                pad_before=pad_before,
                pad_after=pad_after,
            )

    print(f"VTT file saved to: {vtt_file_name}")

In [17]:
# Process the training audio file
process_audio_and_create_vtt(f"data/{train_audio_filename}", "mp3", whisper_model)

100%|██████████| 13623/13623 [00:18<00:00, 724.23frames/s]


DEBUG: Total words transcribed: 231
VTT file saved to: data/raw/train-raw.vtt


In [18]:
# Process the validation audio file
process_audio_and_create_vtt(f"data/{validation_audio_filename}", "mp3", whisper_model)

100%|██████████| 12503/12503 [00:27<00:00, 456.16frames/s]


DEBUG: Total words transcribed: 280
VTT file saved to: data/raw/validation-raw.vtt


### Dataset Cleaning

WARNING: This automation is NOT RECOMMENDED WITHOUT MANUAL PROOF-READING AS IT LEADS TO ERRORS. AN ALTERNATIVE IS TO JUST COPY THE FILES IN THE RAW FOLDER INTO THE DATA FOLDER AND PROOF READ THEM LISTENING TO THE TRANSCRIPT.

train.vtt and validation.vtt files are now present in your folder structure.

You now need to manually (or with the help of openai below) correct that transcript (optionally, with the help of a GPT and the prompt below) and then upload the vtt file to the ADVANCED-transcription repo, into the data folder.

Here is a prompt you can use (but be very careful as the LLM can slightly mess up time-stamps and that breaks transcription):
```
I want your help in correcting a VTT file transcript. I'll provide a list of words that the ASR was not familiar with. Respond, in a code pen, with the contents of the updated VTT file:

<input-VTT>
...
</input-VTT>

<keywords>
...
</keywords>

Respond immediately with the corrected VTT.  Do not run any code.
```

In [20]:
%cd /workspace/ADVANCED-transcription
!uv pip install openai python-dotenv -qU
%cd /workspace/ADVANCED-transcription/speech-to-text

/workspace/ADVANCED-transcription
/workspace/ADVANCED-transcription/speech-to-text


In [21]:
# Securely input your OpenAI API key using getpass
# https://platform.openai.com/api-keys
import os, getpass
from dotenv import load_dotenv

ENV_FILE = ".temp_openai.env"
if os.path.exists(ENV_FILE):
    load_dotenv(ENV_FILE)

api_key = os.getenv("OPENAI_API_KEY") or getpass.getpass("Enter OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

if not os.path.exists(ENV_FILE):
    open(ENV_FILE, "w").write(f"OPENAI_API_KEY={api_key}\n")

Enter OpenAI API key:  ········


In [22]:
import re
from openai import OpenAI
from pydantic import BaseModel
from typing import List, Tuple

client = OpenAI(api_key=api_key)

# Define the structure for the corrected VTT response
class CorrectedTextResponse(BaseModel):
    corrected_text: str

# --- TIMESTAMP DETECTION (fixed) ---
def is_timestamp(line: str) -> bool:
    """
    Check if a line is a VTT timestamp line of the form:
    HH:MM:SS.mmm --> HH:MM:SS.mmm [optional settings...]
    """
    line = line.strip()
    pattern = r"""
        ^\d{2}:\d{2}:\d{2}\.\d{3}      # start time
        \s*-->\s*
        \d{2}:\d{2}:\d{2}\.\d{3}       # end time
        (?:\s+.*)?$                    # optional cue settings
    """
    return re.match(pattern, line, re.VERBOSE) is not None


def get_context(vtt_lines: List[Tuple[str, str]], index: int, context_range: int = 2):
    """Get up to `context_range` cues before and after the current index for context."""
    start_idx = max(0, index - context_range)
    end_idx = min(len(vtt_lines), index + context_range + 1)
    return vtt_lines[start_idx:end_idx]


# --- MULTI-LINE CUES (fixed) ---
def split_vtt_into_lines(vtt_contents: str) -> List[Tuple[str, str]]:
    """
    Splits a VTT file into (timestamp_line, text_block) pairs.
    Supports multi-line cues and ignores header/blank lines.
    """
    lines = vtt_contents.splitlines()
    vtt_pairs: List[Tuple[str, str]] = []

    i = 0
    while i < len(lines):
        line = lines[i].strip()

        # Skip header or empty lines
        if not line or line.upper().startswith("WEBVTT") or line.startswith("NOTE"):
            i += 1
            continue

        if is_timestamp(line):
            timestamp = line

            # Collect all following non-empty, non-timestamp lines as the cue text
            text_lines: List[str] = []
            j = i + 1
            while j < len(lines):
                next_line = lines[j]

                # Blank line = end of cue
                if not next_line.strip():
                    break

                # If this next line is itself a timestamp, treat as start of next cue
                if is_timestamp(next_line.strip()):
                    break

                text_lines.append(next_line.rstrip("\n"))
                j += 1

            text = "\n".join(text_lines).strip()

            # Only add if there is at least some text
            if text:
                vtt_pairs.append((timestamp, text))

            # Move to the line after the cue we just processed
            i = j
        else:
            # Non-timestamp, non-header line – skip
            i += 1

    return vtt_pairs

def correct_single_line(vtt_lines: List[Tuple[str, str]], index_to_correct: int, keywords: str) -> str:
    """Correct a single line with context from the VTT file."""
    # Get up to 2 lines before and after the target line for context
    context_lines = get_context(vtt_lines, index_to_correct)

    # Prepare the input with context
    combined_input = "\n".join(
        [f"{timestamp}\n{text}" for timestamp, text in context_lines]
    )

    # Target line text
    target_timestamp, target_line = vtt_lines[index_to_correct]

    # OPTION 1: To target/fix based on known keywords
    prompt = f"""
I want your help in correcting the following VTT file transcript. I will provide up to two cues before and after the cue that needs correction for context.

Do not modify the timestamps. Do not modify any other lines except the target line of text.

The line that needs correction is marked as <target>:

<input-VTT>
{combined_input}
</input-VTT>

<target>
{target_line}
</target>

<keywords>
{keywords}
</keywords>

Please return only the corrected target line of text (without timestamps and without any extra commentary).
"""

    # # OPTION 2: To target/fix to British English
    #     prompt = f"""
    # I want your help in correcting the following VTT file transcript to British English. I will provide up to two cues before and after the cue that needs correction for context.
    
    # Do not modify the timestamps. Do not modify any other lines except the target line of text.
    
    # The line that needs correction is marked as <target>:
    
    # <input-VTT>
    # {combined_input}
    # </input-VTT>
    
    # <target>
    # {target_line}
    # </target>
    
    # Please return only the corrected target line of text (without timestamps and without any extra commentary).
    # """

    ## DEBUG
    # print(f"Prompt: {prompt}")
    
    try:
        completion = client.beta.chat.completions.parse(
            model="gpt-5-mini",
            reasoning_effort="minimal",
            messages=[
                {
                    "role": "system",
                    "content": "Correct the specified line of text based on the provided context and keywords. Output only the corrected line in the expected JSON schema.",
                },
                {"role": "user", "content": prompt},
            ],
            response_format=CorrectedTextResponse,
        )

        # Defensive extraction
        choice = completion.choices[0]
        if not hasattr(choice.message, "parsed") or choice.message.parsed is None:
            raise ValueError("No parsed structured response returned by the model.")

        corrected_text = choice.message.parsed.corrected_text

        # Final sanity clean
        corrected_text = corrected_text.strip()

        # If the model returned an empty string for some reason, fall back
        if not corrected_text:
            corrected_text = target_line

    except Exception as e:
        print(
            f"Error correcting line {index_to_correct} "
            f"(timestamp {target_timestamp}): {e}. Using original text."
        )
        corrected_text = target_line

    return corrected_text

def correct_vtt_lines(vtt_lines, keywords):
    corrected_lines = []
    for idx, (timestamp, text) in enumerate(vtt_lines):
        print(f"Correcting line {idx + 1} of {len(vtt_lines)}...")
        corrected_text = correct_single_line(vtt_lines, idx, keywords)
        corrected_lines.append((timestamp, corrected_text))

    return corrected_lines

def save_corrected_vtt(corrected_lines, output_file_path):
    """Save the corrected VTT file by combining corrected text with original timestamps."""
    with open(output_file_path, 'w', encoding='utf-8') as file:
        file.write("WEBVTT\n\n")
        for timestamp, text in corrected_lines:
            file.write(f"{timestamp}\n{text}\n\n")

# Main function
def clean_vtt_file(vtt_file_path, keywords=None):
    # Read the VTT file
    with open(vtt_file_path, 'r', encoding='utf-8') as file:
        vtt_contents = file.read()

    # Split VTT file into individual lines (timestamp + text)
    vtt_lines = split_vtt_into_lines(vtt_contents)

    # Correct each line one by one, providing context
    corrected_lines = correct_vtt_lines(vtt_lines, keywords)

    return corrected_lines

# Main function to handle both train and validation files
if __name__ == "__main__":
    train_vtt_path = f"data/raw/{train_audio_filename}-raw.vtt"
    validation_vtt_path = f"data/raw/{validation_audio_filename}-raw.vtt"
    output_train_vtt_path = f"data/{train_audio_filename}.vtt"
    output_validation_vtt_path = f"data/{validation_audio_filename}.vtt"
    keywords_file_name = "data/keywords.txt"
    # keywords_file_name = None

    if keywords_file_name is not None:
        # Load keywords
        with open(keywords_file_name, 'r', encoding='utf-8') as file:
            keywords_to_correct = file.read()
    else:
        keywords_to_correct=None

    # Clean and save the train VTT file
    corrected_train_vtt_lines = clean_vtt_file(train_vtt_path, keywords_to_correct)
    if corrected_train_vtt_lines:
        save_corrected_vtt(corrected_train_vtt_lines, output_train_vtt_path)
        print(f"Corrected Train VTT file saved to: {output_train_vtt_path}")

    # Clean and save the validation VTT file
    corrected_validation_vtt_lines = clean_vtt_file(validation_vtt_path, keywords_to_correct)
    if corrected_validation_vtt_lines:
        save_corrected_vtt(corrected_validation_vtt_lines, output_validation_vtt_path)
        print(f"Corrected Validation VTT file saved to: {output_validation_vtt_path}")

Correcting line 1 of 7...
Correcting line 2 of 7...
Correcting line 3 of 7...
Correcting line 4 of 7...
Correcting line 5 of 7...
Correcting line 6 of 7...
Correcting line 7 of 7...
Corrected Train VTT file saved to: data/train.vtt
Correcting line 1 of 6...
Correcting line 2 of 6...
Correcting line 3 of 6...
Correcting line 4 of 6...
Correcting line 5 of 6...
Correcting line 6 of 6...
Corrected Validation VTT file saved to: data/validation.vtt


### Dataset Creation
You must be logged into huggingface to run this, see above for login.

In [23]:
# %cd /workspace/ADVANCED-transcription/speech-to-text

#### For Processing Multiple Input mp3 and vtt files
Place your files within the `data` folder!

In [69]:
import os
from glob import glob
from datasets import Dataset, DatasetDict, Audio
import webvtt
from datetime import datetime
import librosa
import soundfile as sf

# Setup
data_folder = "data"  # Path to the data folder
save_path = f"data/{dataset_name.split('/')[-1]}-dataset"  # Local save path

def parse_time(time_str):
    return datetime.strptime(time_str, '%H:%M:%S.%f')

def milliseconds(time_obj):
    return (time_obj.hour * 3600 + time_obj.minute * 60 + time_obj.second) * 1000 + time_obj.microsecond // 1000

def time_to_samples(time_ms, sr):
    return int((time_ms / 1000.0) * sr)

def transform_data(data):
    transformed = {"audio": [], "text": [], "start_time": [], "end_time": []}
    for item in data:
        for key in transformed:
            transformed[key].append(item[key])
    return transformed

def process_audio_file(audio_path, vtt_path, output_dir, max_duration=30.0, min_duration=0.0):
    """
    Process a single audio file + its cleaned VTT:
    - Each VTT cue becomes exactly one training example.
    - Chunks with duration <= 0 or > max_duration are skipped.
    """
    os.makedirs(output_dir, exist_ok=True)

    # Load the full audio file
    full_audio, sr = librosa.load(audio_path, sr=None, mono=True)

    # Parse VTT
    captions = webvtt.read(vtt_path)

    data = []
    segment_counter = 0

    for i, caption in enumerate(captions):
        start_time = parse_time(caption.start)
        end_time = parse_time(caption.end)

        duration = (end_time - start_time).total_seconds()

        # Basic sanity checks
        if duration <= 0:
            print(f"[WARN] Skipping caption {i} with non-positive duration: {duration:.3f}s")
            continue

        if duration < min_duration:
            # Optional: skip super-short segments (e.g. coughs, filler)
            print(f"[INFO] Skipping caption {i} shorter than {min_duration}s: {duration:.3f}s")
            continue

        if duration > max_duration:
            # Safety: never produce >30s training clips
            print(f"[WARN] Skipping caption {i} longer than {max_duration}s: {duration:.3f}s")
            continue

        # Convert times to samples
        start_ms = milliseconds(start_time.time())
        end_ms = milliseconds(end_time.time())
        start_sample = time_to_samples(start_ms, sr)
        end_sample = time_to_samples(end_ms, sr)

        if end_sample <= start_sample or end_sample > len(full_audio):
            print(f"[WARN] Skipping caption {i} due to invalid sample range: "
                  f"{start_sample}–{end_sample} (len={len(full_audio)})")
            continue

        # Slice audio and save segment
        segment_filename = os.path.join(output_dir, f"segment_{segment_counter}.mp3")
        audio_segment = full_audio[start_sample:end_sample]
        sf.write(segment_filename, audio_segment, sr, format="mp3")

        # Build dataset entry
        text = caption.text.strip()
        if not text:
            print(f"[INFO] Skipping empty text at caption {i}")
            continue

        data.append({
            "audio": segment_filename,
            "text": text,
            "start_time": start_time.strftime('%H:%M:%S.%f')[:-3],
            "end_time":   end_time.strftime('%H:%M:%S.%f')[:-3],
        })

        segment_counter += 1

    return data

def process_files(file_type):
    audio_files = glob(os.path.join(data_folder, f"*{file_type}*.mp3"))
    vtt_files = glob(os.path.join(data_folder, f"*{file_type}*.vtt"))

    if not audio_files or not vtt_files:
        raise ValueError(f"No {file_type} files found in the data folder.")

    # Optional: sort for deterministic pairing
    audio_files.sort()
    vtt_files.sort()

    data = []
    for audio_file, vtt_file in zip(audio_files, vtt_files):
        data.extend(process_audio_file(audio_file, vtt_file, f"{save_path}/{file_type}"))

    return data

def create_dataset(output_dir):
    os.makedirs(output_dir, exist_ok=True)

    # Process training files
    train_data = process_files("train")

    # Process validation files
    validation_data = process_files("validation")

    # Transform data into the correct format for Dataset.from_dict
    train_dataset = Dataset.from_dict(transform_data(train_data))
    valid_dataset = Dataset.from_dict(transform_data(validation_data))

    # Create DatasetDict
    dataset_dict = DatasetDict({
        "train": train_dataset,
        "validation": valid_dataset
    })

    return dataset_dict

# Create dataset
dataset = create_dataset(save_path)

# Save dataset locally
dataset.save_to_disk(save_path)

# Cast the audio column to the Audio feature
dataset = dataset.cast_column("audio", Audio())

# Push the dataset to the Hub
dataset.push_to_hub(repo_id=dataset_name)

Saving the dataset (0/1 shards):   0%|          | 0/7 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/6 [00:00<?, ? examples/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/Trelis/llm-lingo/commit/485d553e213629a7c209c183e47325f328a5859e', commit_message='Upload dataset', commit_description='', oid='485d553e213629a7c209c183e47325f328a5859e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Trelis/llm-lingo', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Trelis/llm-lingo'), pr_revision=None, pr_num=None)

## Training and Evaluation
RESTART YOUR KERNEL BEFORE RUNNING THESE CELLS

### Load a Pre-Trained Checkpoint

In [70]:
# # owing to an issue with a new chat template structure, you may need to run this
# !uv pip install -qU transformers huggingface_hub

In [71]:
from unsloth import FastModel
from transformers import WhisperForConditionalGeneration
import torch

# del model

model, tokenizer = FastModel.from_pretrained(
    model_name = model_name_or_path,
    dtype = None, # Leave as None for auto detection
    load_in_4bit = False, # Set to True to do 4bit quantization which reduces memory (and accuracy a bit)
    auto_model = WhisperForConditionalGeneration,
    whisper_language = "English",
    whisper_task = "transcribe"
)

==((====))==  Unsloth 2025.11.3: Fast Whisper patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A40. Num GPUs = 1. Max memory: 44.339 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.
unsloth/whisper-large-v3-turbo does not have a padding token! Will use pad_token = <|endoftext|>.


In [72]:
# print(model)

In [73]:
model = FastModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    task_type = None, # ** MUST set this for Whisper **
)

Unsloth: Making `model.base_model.model.model.encoder` require gradients


Override generation arguments:
- no tokens are forced as decoder outputs (see [`forced_decoder_ids`](https://huggingface.co/docs/transformers/main_classes/text_generation#transformers.generation_utils.GenerationMixin.generate.forced_decoder_ids))
- no tokens are suppressed during generation (see [`suppress_tokens`](https://huggingface.co/docs/transformers/main_classes/text_generation#transformers.generation_utils.GenerationMixin.generate.suppress_tokens))

### Load Dataset

In [74]:
import numpy as np
import tqdm
from datasets import DatasetDict

#Set this to the language you want to train on
model.generation_config.language = "<|en|>"
model.generation_config.task = "transcribe"
model.config.suppress_tokens = []
model.generation_config.forced_decoder_ids = None

def formatting_prompts_func(example):
    audio_arrays = example['audio']['array']
    sampling_rate = example["audio"]["sampling_rate"]
    features = tokenizer.feature_extractor(
        audio_arrays, sampling_rate=sampling_rate
    )
    tokenized_text = tokenizer.tokenizer(example["text"])
    return {
        "input_features": features.input_features[0],
        "labels": tokenized_text.input_ids,
    }
    
from datasets import load_dataset, Audio
dataset = DatasetDict()
dataset["train"] = load_dataset(dataset_name, split="train")
dataset["validation"] = load_dataset(dataset_name, split="validation")

dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
train_dataset = [formatting_prompts_func(example) for example in tqdm.tqdm(dataset['train'], desc='Train split')]
validation_dataset = [formatting_prompts_func(example) for example in tqdm.tqdm(dataset['validation'], desc='Validation split')]

README.md:   0%|          | 0.00/494 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.24M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6 [00:00<?, ? examples/s]

Validation split: 100%|██████████| 6/6 [00:00<00:00, 10.36it/s]


In [75]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['audio', 'text', 'start_time', 'end_time'],
        num_rows: 7
    })
    validation: Dataset({
        features: ['audio', 'text', 'start_time', 'end_time'],
        num_rows: 6
    })
})


### Define a Data Collator

In [76]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:

        # Split inputs and labels since they have to be of different lengths and need different padding methods
        # First treat the audio inputs (input features)
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Get the tokenized label sequences (input_ids)
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # Pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # If bos token is appended in the previous tokenization step, remove it here
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        # IMPORTANT: Do NOT cast input_ids/labels to bfloat16; they should remain as Long tensors
        batch["labels"] = labels  # labels remain in their original type (Long)

        return batch

Let's initialise the data collator we've just defined:

In [77]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=tokenizer)

### Evaluation Metrics

We'll use the word error rate (WER) metric, the 'de-facto' metric for assessing
ASR systems. For more information, refer to the WER [docs](https://huggingface.co/metrics/wer). We'll load the WER metric from 🤗 Evaluate:

In [78]:
import evaluate

metric = evaluate.load("wer")

We then simply have to define a function that takes our model
predictions and returns the WER metric. This function, called
`compute_metrics`, first replaces `-100` with the `pad_token_id`
in the `label_ids` (undoing the step we applied in the
data collator to ignore padded tokens correctly in the loss).
It then decodes the predicted and label ids to strings. Finally,
it computes the WER between the predictions and reference labels:

In [ ]:
import numpy as np

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    
    # If using teacher forcing (predict_with_generate=False), predictions are logits (3D: batch x seq x vocab)
    # We need to take argmax to get predicted token IDs
    if len(pred_ids.shape) == 3:
        pred_ids = np.argmax(pred_ids, axis=-1)

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

Now let's load the pre-trained Whisper `turbo` checkpoint.

### Define the Training Configuration

In the final step, we define all the parameters related to training. For more detail on the training arguments, refer to the Seq2SeqTrainingArguments [docs](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Seq2SeqTrainingArguments).

In [ ]:
from transformers import Seq2SeqTrainingArguments
from unsloth import is_bf16_supported

# Toggle for teacher forcing vs generation during evaluation
# False = use generation (more realistic, higher WER due to error accumulation)
# True = use teacher forcing (lower WER, but less realistic as model sees ground truth at each step)
use_teacher_forcing = False

training_args = Seq2SeqTrainingArguments(
    output_dir=f"outputs/{trained_model_name}",  # change to a repo name of your choice
    per_device_train_batch_size=1, # use 1 if you have a dataset of just 10. Use 4 if you've a dataset of 50-200, use 16 for larger than that, or 32.
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,  # probably best to leave at 1 unless you see a lot of noise on the training loss.
    learning_rate=1e-4,
    # warmup_steps=50,
    num_train_epochs=2,
    eval_strategy="steps",
    fp16=not is_bf16_supported,
    bf16=is_bf16_supported,
    # generation_max_length=128,
    logging_steps=1,
    remove_unused_columns=False,  # required as the PeftModel forward doesn't have the signature of the wrapped model's forward
    label_names=["labels"],  # same reason as above
    predict_with_generate=not use_teacher_forcing,  # generation=True gives more realistic WER; teacher forcing=False gives lower WER
    save_steps=0, #if you wish to save checkpoints, select how many steps to do this on
    eval_steps=2, # run every two steps
    do_eval=True,
    lr_scheduler_type="constant",
)

**Important Notes:**
1. `remove_unused_columns=False` and `label_names=["labels"]` are required as the PeftModel's forward doesn't have the signature of the base model's forward.

2. If using INT8 - INT8 training required autocasting. `predict_with_generate` can't be passed to Trainer because it internally calls transformer's `generate` without autocasting leading to errors.

3. If using INT8 - Because of point 2, `compute_metrics` shouldn't be passed to `Seq2SeqTrainer` as seen below. (commented out)

In [81]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer.feature_extractor,
)
model.config.use_cache = False  # silence the warnings. Please re-enable for inference!

/tmp/ipykernel_5962/1808072967.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [82]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7 | Num Epochs = 2 | Total steps = 14
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 27,852,800 of 836,730,880 (3.33% trained)


GPU = NVIDIA A40. Max memory = 44.339 GB.
13.748 GB of memory reserved.


Step,Training Loss,Validation Loss,Wer
2,2.430600,1.037305,36.073059
4,0.642700,0.781926,21.461187
6,0.805900,0.589829,20.547945
8,0.202800,0.514202,19.634703
10,0.290600,0.487088,19.178082
12,0.132800,0.427132,19.178082
14,0.502400,0.419782,19.178082


In [83]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

36.8009 seconds used for training.
0.61 minutes used for training.
Peak reserved memory = 13.748 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 31.007 %.
Peak reserved memory for training % of max memory = 0.0 %.


### Push adapters to hub and merge adapters onto the model

In [84]:
# WARNING - you may be missing generation_config, which might need to grab from the base model and add to your repo!

# if not already in this folder
%cd "speech-to-text"

# Merge to 16bit
if False: model.save_pretrained_merged(trained_model_name, tokenizer, save_method = "merged_16bit")
if True: model.push_to_hub_merged(trained_model_repo, tokenizer, save_method = "merged_16bit")

# Merge to 4bit
if False: model.save_pretrained_merged(trained_model_name, tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged(trained_model_repo, tokenizer, save_method = "merged_4bit")

# Just LoRA adapters
if False:
    model.save_pretrained("lora-adapters")
    tokenizer.save_pretrained("lora-adapters")
if False:
    model.push_to_hub(trained_adapter_repo)
    tokenizer.push_to_hub(trained_adapter_repo)

[Errno 2] No such file or directory: 'speech-to-text'
/workspace/ADVANCED-transcription/speech-to-text


No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `Trelis/llm-lingo`: 100%|██████████| 1/1 [00:05<00:00,  5.43s/it]


Successfully copied all 1 files from cache to `Trelis/llm-lingo`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:23<00:00, 23.48s/it]


Unsloth: Merge process complete. Saved to `/workspace/ADVANCED-transcription/speech-to-text/Trelis/llm-lingo`


In [85]:
# # Check the LoRa adapters are merged
# print(model)
print(model.dtype)  # should match input features

torch.bfloat16


## Evaluation

In [86]:
import torch
from transformers import pipeline

audio = "data/validation.mp3"

def run_asr(model_or_repo, tokenizer):
    return pipeline(
        "automatic-speech-recognition",
        model=model_or_repo,
        tokenizer=tokenizer.tokenizer,
        feature_extractor=tokenizer.feature_extractor,
        processor=tokenizer,
        return_language=True,
        return_timestamps="word",
        torch_dtype=torch.bfloat16 if is_bf16_supported else torch.float16,
    )(audio)["text"]

# --- Base (untrained) model ---
unsloth_text = run_asr(model_name_or_path, tokenizer)

# --- Trained model ---
trained_text = run_asr(trained_model_repo, tokenizer)

print("\n=== Untrained ===\n", unsloth_text)
print("\n=== Trained ===\n", trained_text)

Device set to use cuda:0


model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Device set to use cuda:0



=== Untrained ===
  MMLU is a means of testing performance. Claude 2.0 is the latest Claude model from Anthropic. Nodux 8x7bv1 is a fine tune of Mixtral. Wizard LM70b is a fine tune of Lama 70b, it's actually Wizard LM70bv1.0. spin self-play fine tuning that improves llms the sclm 7b is a fast model with 7 billion parameters arena elo is a means of measuring performance the fastest open ai model is gpt4 turbo open chat is a fine tune then um basically it's a fine tune of the mistral 7b model tricksy is an approach for fast inference using sparsity microsoft have launched phi2 mt bench is a metric for performance eval mistral medium is a larger mixture of experts claude one is an earlier version of claude from anthropic mixed rally by 7b instruct v 0.1 is the mixture of experts with 7b models to loot 2 dpo is a fine-tune of the 70b model gemini pro is google's best model solar 10.7b is the fastest uh rather sorry it's the it's a strong it's a version of mistral 7b with extra layers cla

In [87]:
# ## Run this code if you face utf locale errors in Colab
# # https://stackoverflow.com/questions/56081324/why-are-google-colab-shell-commands-not-working
# import locale
# def getpreferredencoding(do_setlocale = True):
#     return "UTF-8"
# locale.getpreferredencoding = getpreferredencoding

## Convert HuggingFace Whisper Model to OpenAI format

In [88]:
STOP HERE

!pip install git+https://github.com/openai/whisper.git -q

SyntaxError: invalid syntax (1685254524.py, line 1)

In [ ]:
# Load the model in HuggingFace format
from transformers import WhisperForConditionalGeneration

# # del model # if you want to re-load a model
# trained_model_name = "whisper-turbo-llm-lingo" # comment this in and ensure correct if you have restarted the kernel
# trained_model_repo = "Trelis/whisper-turbo-llm-lingo"

model = WhisperForConditionalGeneration.from_pretrained(
    trained_model_name,
    load_in_8bit=False,
    torch_dtype=torch.bfloat16 if use_bf16 else torch.float16, # or float16 for Colab
    device_map="auto")

In [ ]:
trained_model_bin_format = f"{trained_model_name}-bin"

model.save_pretrained(trained_model_bin_format,
                      safe_serialization=False # to save to a pickled format
                     )
# processor.save_pretrained(trained_model_bin_format) # not necessary

In [ ]:
# Credit to [ndunks](https://github.com/ndunks) for the snippet
#!/bin/env python3
import whisper
import re
import torch

def hf_to_whisper_states(text):
    text = re.sub('.layers.', '.blocks.', text)
    text = re.sub('.self_attn.', '.attn.', text)
    text = re.sub('.q_proj.', '.query.', text)
    text = re.sub('.k_proj.', '.key.', text)
    text = re.sub('.v_proj.', '.value.', text)
    text = re.sub('.out_proj.', '.out.', text)
    text = re.sub('.fc1.', '.mlp.0.', text)
    text = re.sub('.fc2.', '.mlp.2.', text)
    text = re.sub('.fc3.', '.mlp.3.', text)
    text = re.sub('.fc3.', '.mlp.3.', text)
    text = re.sub('.encoder_attn.', '.cross_attn.', text)
    text = re.sub('.cross_attn.ln.', '.cross_attn_ln.', text)
    text = re.sub('.embed_positions.weight', '.positional_embedding', text)
    text = re.sub('.embed_tokens.', '.token_embedding.', text)
    text = re.sub('model.', '', text)
    text = re.sub('attn.layer_norm.', 'attn_ln.', text)
    text = re.sub('.final_layer_norm.', '.mlp_ln.', text)
    text = re.sub('encoder.layer_norm.', 'encoder.ln_post.', text)
    text = re.sub('decoder.layer_norm.', 'decoder.ln.', text)
    text = re.sub('proj_out.weight', 'decoder.token_embedding.weight', text)
    return text

# Load HF Model
hf_state_dict = torch.load(f"{trained_model_bin_format}/pytorch_model.bin", map_location=torch.device('cpu'))    # pytorch_model.bin file

# Rename layers
for key in list(hf_state_dict.keys())[:]:
    new_key = hf_to_whisper_states(key)
    hf_state_dict[new_key] = hf_state_dict.pop(key)

openai_format_model = f"{trained_model_name}-openai.bin"

model = whisper.load_model('turbo')
dims = model.dims
# Save it
torch.save({
    "dims": model.dims.__dict__,
    "model_state_dict": hf_state_dict
}, openai_format_model)

In [ ]:
## Upload to hub
from huggingface_hub import upload_file

openai_format_model = f"{trained_model_name}-openai.bin"
trained_model_repo = "Trelis/whisper-turbo-llm-lingo"

# Upload the file to the repository
upload_file(
    path_or_fileobj=openai_format_model,  # The file path or object
    path_in_repo=openai_format_model,  # The name under which to save it in the repo
    repo_id=trained_model_repo,  # The repo ID
    commit_message=f"Add {openai_format_model}",  # Optional: commit message
    commit_description="Uploading the OpenAI format bin model",  # Optional: description
    create_pr=False  # Set to True if you want to create a PR instead of pushing to the main branch
)

## Converting HuggingFace format to CTranslate2 (for Faster Whisper)

In [ ]:
!pip install ctranslate2 -qU
# !pip uninstall ctranslate2 -y

In [ ]:
trained_model_path = "model_name_or_path"
local_ctranslate_repo = f"{trained_model_repo.split('/')[-1]}-ctranslate2"
org = "Trelis"
hf_ctranslate_repo = f"{org}/{local_ctranslate_repo}"

In [ ]:
## IMPORTANT: To run this, you'll need to grab the tokenizer.json from
# https://huggingface.co/openai/whisper-large-v3-turbo/tree/main and
# then upload it to where you saved your trained model locally

# Convert to float16 and copy the necessary tokenizer files
!ct2-transformers-converter --model "{trained_model_path}" --output_dir "{local_ctranslate_repo}" --copy_files tokenizer.json preprocessor_config.json --quantization float16

In [ ]:
## Upload to hub
from huggingface_hub import upload_folder, create_repo

create_repo(hf_ctranslate_repo, repo_type="model")

# Upload the file to the repository
upload_folder(
    folder_path=local_ctranslate_repo,  # The file path or object
    repo_id=hf_ctranslate_repo,  # The repo ID
    commit_message=f"Add {local_ctranslate_repo}",  # Optional: commit message
    commit_description="Uploading the ctranslate format model",  # Optional: description
    create_pr=False  # Set to True if you want to create a PR instead of pushing to the main branch
)

### Quick test of the ctranslate2 model

In [ ]:
import ctranslate2
import librosa
import transformers

# Load and resample the audio file.
audio, _ = librosa.load("validation.mp3", sr=16000, mono=True) # you need to upload some mp3 (or wav or mp4) audio and name it validation.xxx

# Compute the features of the first 30 seconds of audio.
## IMPORTANT - THE PROCESSOR HERE IS BEING LOADED FROM THE HUGGINGFACE SAVED MODEL
processor = transformers.WhisperProcessor.from_pretrained(trained_model_repo)  # local directory
inputs = processor(audio, return_tensors="np", sampling_rate=16000)

# Convert the input features to the StorageView type for ctranslate2.
features = ctranslate2.StorageView.from_array(inputs.input_features)

# Load the model.
model = ctranslate2.models.Whisper(local_ctranslate_repo)

# Detect the language.
results = model.detect_language(features)
language, probability = results[0][0]
print("Detected language %s with probability %f" % (language, probability))

# Describe the task in the prompt.
prompt = processor.tokenizer.convert_tokens_to_ids(
    [
        "<|startoftranscript|>",
        language,
        "<|transcribe|>",
        "<|notimestamps|>",  # Remove this token to generate timestamps.
    ]
)

# Run generation for the 30-second window.
results = model.generate(features, [prompt])
transcription = processor.decode(results[0].sequences_ids[0])
print(transcription)